In [ ]:
from pathlib import Path
import os
import sys

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'config/paper.json').is_file())
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator
plt.rcdefaults()
out = Path('Figures/Generated')
out.mkdir(parents=True, exist_ok=True)
timezone = 'America/New_York'
plt.rcParams.update({'text.usetex': True, 'font.family': 'serif', 'font.serif': ['Palatino'], 'text.latex.preamble': '\\usepackage[T1]{fontenc}'})
reader = json.loads(Path('FigureSources/news_candidates.json').read_text())
supervisor = json.loads(Path('FigureSources/news_selection.json').read_text())
candidates = pd.DataFrame(reader['candidates'], columns=reader['candidate_columns'])
candidates['published'] = pd.to_datetime(candidates.published_at_utc, utc=True).dt.tz_convert(timezone)
selected = pd.DataFrame(supervisor['news_fact_context']['articles']).sort_values('published_at').reset_index(drop=True)
selected['published'] = pd.to_datetime(selected.published_at, utc=True).dt.tz_convert(timezone)
selected['letter'] = list('ABCDEFG')
short = {2106: 'Heathrow estimates £1bn for a more resilient power supply', 2216: 'UK banks challenge the car-loan commission ruling', 2563: 'Tesla quarterly sales fall 13% amid the Musk backlash', 2572: 'Gold rises as markets await the US tariff announcement', 2630: 'Sweeping US tariffs announced as Musk’s role draws attention', 2634: 'Australia offers $1bn in loans to affected exporters', 2657: 'Trump’s sweeping tariffs reshape US trade policy'}
selected['short_title'] = selected['index'].map(short)
window = supervisor['window']['news_window']
start, end = (pd.Timestamp(window['start_utc']).tz_convert(timezone), pd.Timestamp(window['end_utc']).tz_convert(timezone))
assert len(candidates) == 44 and len(selected) == 7 and selected['index'].isin(candidates['index']).all()
assert candidates.published.ge(start).all() and candidates.published.lt(end).all()
assert start.strftime('%Y-%m-%d %H:%M %z') == '2025-03-05 09:30 -0500'
assert end.strftime('%Y-%m-%d %H:%M %z') == '2025-04-03 09:30 -0400'
fig = plt.figure(figsize=(6.8, 3.5))
fig.patch.set_facecolor('white')
ax = fig.add_axes([0.065, 0.78, 0.9, 0.085])
ax.set_facecolor('white')
ax.axvspan(start, end, color='#4169E1', alpha=0.07, zorder=0)
ax.scatter(candidates.published, np.full(len(candidates), 0.5), marker='|', s=110, c='#89929D', linewidths=0.7, zorder=2)
ax.scatter(selected.published, np.full(len(selected), 0.5), s=20, c='#4169E1', edgecolors='white', linewidths=0.5, zorder=3)
ax.axhline(0.5, color='#bac3cc', lw=0.5, zorder=1)
ax.set_ylim(0, 1)
ax.set_xlim(start - pd.Timedelta(hours=12), end + pd.Timedelta(hours=12))
ax.set_yticks([])
ticks = pd.DatetimeIndex(['2025-03-05 09:30', '2025-03-12 09:30', '2025-03-19 09:30', '2025-03-26 09:30', '2025-04-03 09:30'], tz=timezone)
ax.set_xticks(ticks, ['05 Mar', '12 Mar', '19 Mar', '26 Mar', '03 Apr'])
ax.tick_params(axis='x', labelsize=9, length=0, pad=5)
fig.text(0.065, 0.968, '\\textbf{S\\&P 500 · 03 April 2025, before 09:30 New York time}', fontsize=10.5, va='top')
fig.text(0.065, 0.67, '\\textbf{Selected article}', fontsize=10, color='black')
fig.text(0.965, 0.67, '\\textbf{Publication time (New York)}', fontsize=9, color='black', ha='right')
fig.canvas.draw()
actual_positions = ax.transData.transform(np.column_stack([mdates.date2num(selected.published), np.full(len(selected), 0.5)]))[:, 0]
label_positions = actual_positions.copy()
minimum_gap = 10 * fig.dpi / 72
for i in range(len(label_positions) - 2, -1, -1):
    label_positions[i] = min(label_positions[i], label_positions[i + 1] - minimum_gap)
timeline_labels = []
for row, actual, label in zip(selected.itertuples(index=False), actual_positions, label_positions):
    timeline_labels.append(ax.annotate(row.letter, xy=(row.published, 0.5), xytext=((label - actual) * 72 / fig.dpi, 14), textcoords='offset points', ha='center', va='bottom', fontsize=9, color='#4169E1', annotation_clip=False, arrowprops={'arrowstyle': '-', 'color': '#4169E1', 'lw': 0.5, 'shrinkA': 1.5, 'shrinkB': 3}))
row_text = []
for i, row in enumerate(selected.itertuples(index=False)):
    y = 0.595 - i * 0.076
    fig.text(0.065, y, row.letter, va='center', ha='center', fontsize=9, color='white', bbox={'boxstyle': 'circle,pad=.22', 'facecolor': '#4169E1', 'edgecolor': 'none'})
    escaped = row.short_title.replace('&', '\\&').replace('%', '\\%').replace('$', '\\$').replace('£', '\\pounds{}')
    title = fig.text(0.095, y, escaped, fontsize=10, va='center')
    stamp = fig.text(0.965, y, row.published.strftime('%d %b · %H:%M'), fontsize=9, ha='right', va='center', color='black')
    row_text.append((title, stamp))
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
for title, stamp in row_text:
    assert title.get_window_extent(renderer).x1 + 8 < stamp.get_window_extent(renderer).x0, (title.get_text(), title.get_window_extent(renderer).bounds, stamp.get_window_extent(renderer).bounds)
for text in [*fig.texts, *ax.get_xticklabels(), *timeline_labels]:
    box = text.get_window_extent(renderer)
    assert fig.bbox.contains(*box.get_points()[0]) and fig.bbox.contains(*box.get_points()[1]), text.get_text()
assert all((spine.get_visible() for spine in ax.spines.values()))
assert np.all(np.diff(label_positions) >= minimum_gap - 0.01)
fig.savefig(out / 'Figure 07 News Selection.png', dpi=360, facecolor='white')
plt.show()
